# Phase 3 - Notebook 03: Global Alignment from Pairwise Poses

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase3/03_global_alignment.ipynb)


## Setup


In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from src.dust3r.alignment import GlobalAligner
from src.dust3r.pointmap import PointMap
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

print("Setup complete!")

## 1. 问题：从配对位姿到全局位姿

给定 N 张图像，DUSt3R 可以对每对图像估计相对位姿。

**问题**：如何从这些配对的相对位姿估计全局位姿？

```
图像1 -- T_{12} --> 图像2
  |                    |
  | T_1                | T_2
  ↓                    ↓
全局坐标系

目标：找到 T_1, T_2, ..., T_N 使得所有配对约束 T_ij 一致
```


In [ ]:
# 创建合成的多视图场景
aligner = GlobalAligner()

# 创建 3 个 Pointmaps
for i in range(3):
    H, W = 16, 16
    u, v = np.meshgrid(np.arange(W), np.arange(H))
    
    # 添加视点特定的扰动
    angle = i * np.pi / 6
    R = np.array([
        [np.cos(angle), -np.sin(angle), 0],
        [np.sin(angle), np.cos(angle), 0],
        [0, 0, 1]
    ])
    
    # 基础深度
    depth = 2.0 + 0.5 * np.sin(u / 8) + 0.3 * np.cos(v / 8)
    
    # 创建 3D 点
    fx = fy = 50.0
    cx = cy = W / 2
    X = (u - cx) * depth / fx
    Y = (v - cy) * depth / fy
    Z = depth
    
    points = np.stack([X, Y, Z], axis=-1)
    points_transformed = (points @ R.T) + np.array([i, 0, 0])[None, None, :]
    
    confidence = 0.7 + 0.3 * np.random.rand(H, W)
    pm = PointMap(points_transformed, confidence=confidence)
    aligner.add_frame(f'frame_{i}', pm)

print(f"添加了 {len(aligner.points)} 个帧到 GlobalAligner")

## 2. Procrustes 分析：点云对齐

**Procrustes 问题**：给定两个点集 P_src 和 P_tgt，找到最优的旋转 R 和平移 t 使得 P_src 与 P_tgt 对齐。

通过 SVD 求解：
1. 中心化两个点集
2. SVD 分解
3. 构造旋转矩阵


In [ ]:
# 演示 Procrustes 对齐
from src.dust3r.alignment import GlobalAligner

# 创建两个相似的点集
P_src = np.random.randn(100, 3) * 2

# 应用已知变换
angle = np.radians(30)
R_true = np.array([
    [np.cos(angle), -np.sin(angle), 0],
    [np.sin(angle), np.cos(angle), 0],
    [0, 0, 1]
])
t_true = np.array([1, 2, 0.5])

P_tgt = (P_src @ R_true.T) + t_true[None, :]

# 使用 Procrustes 恢复变换
R_est, t_est = GlobalAligner.procrustes_align(P_src, P_tgt)

print("已知变换：")
print(f"R_true:\n{R_true}")
print(f"t_true: {t_true}")
print()
print("估计的变换：")
print(f"R_est:\n{R_est}")
print(f"t_est: {t_est}")
print()
print(f"旋转误差 (Frobenius): {np.linalg.norm(R_true - R_est):.6f}")
print(f"平移误差: {np.linalg.norm(t_true - t_est):.6f}")

## 3. 多视图全局对齐


In [ ]:
# 创建配对的相对位姿
pairwise_poses = {}

# 生成相对位姿（通常来自 DUSt3R）
for i in range(3):
    for j in range(i+1, 3):
        # 简化：使用已知的变换作为相对位姿
        angle = (j - i) * np.pi / 6
        R_rel = np.array([
            [np.cos(angle), -np.sin(angle), 0],
            [np.sin(angle), np.cos(angle), 0],
            [0, 0, 1]
        ])
        t_rel = np.array([j - i, 0, 0])
        pairwise_poses[(f'frame_{i}', f'frame_{j}')] = (R_rel, t_rel)

print(f"配对相对位姿数量: {len(pairwise_poses)}")
for (i, j), (R, t) in pairwise_poses.items():
    print(f"  {i} → {j}: R det={np.linalg.det(R):.3f}, t={t}")

In [ ]:
# 计算全局位姿
global_poses = aligner.pairwise_to_global(pairwise_poses)

print("全局位姿：")
for frame_id, (R, t) in global_poses.items():
    print(f"  {frame_id}: t={t}")

## 4. 场景尺度估计


In [ ]:
# 估计场景尺度
scale = aligner.compute_scene_scale(
    {f'frame_{i}': aligner.points[f'frame_{i}'] for i in range(3)},
    {f'frame_{i}': aligner.points[f'frame_{i}'] for i in range(3)}
)

print(f"场景尺度: {scale:.3f}")

## 5. 可视化全局点云


In [ ]:
# 可视化所有帧的点云
fig = plt.figure(figsize=(14, 6))

ax = fig.add_subplot(111, projection='3d')

colors = ['red', 'green', 'blue']
for i, frame_id in enumerate(aligner.points):
    pm = aligner.points[frame_id]
    pts = pm.points.reshape(-1, 3)
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], 
              c=colors[i % len(colors)], label=frame_id, s=10, alpha=0.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('全局对齐的多视图点云')
ax.legend()
ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig('global_alignment.png', dpi=100, bbox_inches='tight')
plt.show()

print("所有帧的点云现在在全局坐标系中对齐！")

## 6. Summary

**Key Concepts:**
1. Procrustes 分析用于点云对齐
2. 配对位姿可转换为全局位姿
3. 多个 Pointmaps 可合并为全局点云
4. 场景尺度从点间距估计

---

**Next**: [04_dust3r_architecture.ipynb](./04_dust3r_architecture.ipynb)
